<a href="https://colab.research.google.com/github/RushabhMowade/AQI-Forecasting/blob/main/AQI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Load a sneak peek of your files
df1 = pd.read_csv('Air_quality_data.csv', nrows=5)
df2 = pd.read_csv('aqi_dataset.csv', nrows=5)

print("--- Columns in Air_quality_data.csv ---")
print(df1.columns.tolist())

print("\n--- Columns in aqi_dataset.csv ---")
print(df2.columns.tolist())

--- Columns in Air_quality_data.csv ---
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'AQI', 'AQI_Bucket']

--- Columns in aqi_dataset.csv ---
['temperature', 'humidity', 'wind_speed', 'traffic_density', 'industrial_activity', 'AQI']


In [3]:
import pandas as pd
import numpy as np

print("⏳ Step 1: Loading your Colab files...")
air_quality_df = pd.read_csv('Air_quality_data.csv')
weather_df = pd.read_csv('aqi_dataset.csv')

# Drop the duplicate AQI column from the weather dataset to prevent a conflict (AQI_x, AQI_y)
if 'AQI' in weather_df.columns:
    weather_df = weather_df.drop(columns=['AQI'])

print("🔗 Step 2: Combining datasets side-by-side...")
# Concatenate horizontally along columns
final_df = pd.concat([air_quality_df, weather_df], axis=1)

# Enforce uniform Datetime formatting and sort chronologically per city
final_df['Datetime'] = pd.to_datetime(final_df['Datetime'])
final_df = final_df.sort_values(by=['City', 'Datetime']).reset_index(drop=True)

print("🛠️ Step 3: Cleaning gaps using Linear Interpolation...")
numeric_cols = final_df.select_dtypes(include=[np.number]).columns
final_df[numeric_cols] = final_df.groupby('City')[numeric_cols].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

print("⏮️ Step 4: Generating temporal Lag Features for previous hours...")
# Creating lag columns based on the true AQI column
final_df['AQI_lag_1h'] = final_df.groupby('City')['AQI'].shift(1)
final_df['AQI_lag_2h'] = final_df.groupby('City')['AQI'].shift(2)
final_df['AQI_lag_24h'] = final_df.groupby('City')['AQI'].shift(24)

# Drop rows at the beginning of each city timeline that have empty lag spaces
final_df.dropna(subset=['AQI_lag_1h', 'AQI_lag_2h', 'AQI_lag_24h'], inplace=True)

print(f"\n🎉 Success! Master table ready. Shape: {final_df.shape}")
print("\nYour final columns for training are:")
print(final_df.columns.tolist())

⏳ Step 1: Loading your Colab files...
🔗 Step 2: Combining datasets side-by-side...
🛠️ Step 3: Cleaning gaps using Linear Interpolation...
⏮️ Step 4: Generating temporal Lag Features for previous hours...

🎉 Success! Master table ready. Shape: (18145, 21)

Your final columns for training are:
['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'AQI', 'AQI_Bucket', 'temperature', 'humidity', 'wind_speed', 'traffic_density', 'industrial_activity', 'AQI_lag_1h', 'AQI_lag_2h', 'AQI_lag_24h']


In [4]:
final_df.head(10)

,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,AQI,AQI_Bucket,temperature,humidity,wind_speed,traffic_density,industrial_activity,AQI_lag_1h,AQI_lag_2h,AQI_lag_24h
24,Bangalore,2015-01-25,219.2,22.7,7.2,121.4,75.0,31.6,6.53,95.3,...,376.4,Very Poor,29.11,46.03,7.65,1.0,58.96,340.8,175.8,339.8
25,Bangalore,2015-01-26,274.2,119.4,78.8,34.3,35.9,19.5,0.31,79.3,...,424.2,Severe,21.48,40.95,9.62,9.0,31.98,376.4,340.8,267.5
26,Bangalore,2015-01-27,469.0,203.6,6.7,120.8,224.1,27.3,4.59,65.7,...,169.2,Moderate,18.56,43.17,8.86,9.0,63.96,424.2,376.4,419.3
27,Bangalore,2015-01-28,447.1,152.5,38.3,131.8,91.2,12.1,8.66,30.7,...,209.4,Poor,16.38,71.27,11.19,9.0,43.54,169.2,424.2,245.7
28,Bangalore,2015-01-29,258.8,95.3,145.4,77.9,13.3,13.3,6.48,16.4,...,408.8,Severe,26.82,72.80,7.04,6.0,48.11,209.4,169.2,413.8
29,Bangalore,2015-01-30,17.6,418.7,166.1,62.8,46.6,44.5,8.19,85.4,...,385.8,Very Poor,27.08,81.77,5.98,9.0,58.81,408.8,209.4,118.9
30,Bangalore,2015-01-31,327.5,71.1,73.7,105.1,203.7,3.0,6.85,4.4,...,477.5,Severe,27.05,50.60,4.40,7.0,108.87,385.8,408.8,302.5
31,Bangalore,2015-02-01,245.6,397.9,73.9,91.1,212.8,46.1,1.84,59.0,...,396.6,Very Poor,29.60,64.44,10.47,4.0,66.10,477.5,385.8,192.0
32,Bangalore,2015-02-02,265.2,8.8,63.5,97.5,21.2,15.6,1.71,41.5,...,415.2,Severe,31.74,63.69,12.67,10.0,79.90,396.6,477.5,220.7
33,Bangalore,2015-02-03,413.9,371.0,153.5,37.8,117.5,6.5,8.55,56.1,...,326.1,Very Poor,19.72,49.87,11.88,7.0,44.96,415.2,396.6,443.8


In [5]:
import pandas as pd
import numpy as np

# 1. Cyclical Encoding for Hour of the Day
print("🔄 Step 5: Applying Cyclical Encoding to Datetime...")
final_df['Hour'] = final_df['Datetime'].dt.hour
final_df['Hour_sin'] = np.sin(2 * np.pi * final_df['Hour'] / 24.0)
final_df['Hour_cos'] = np.cos(2 * np.pi * final_df['Hour'] / 24.0)

# 2. Chronological Splitting
print("📅 Step 6: Splitting data chronologically...")
# We use the final 20% of our timeline as the test set
split_idx = int(len(final_df) * 0.8)

# Features we don't want to feed directly into the ML model equations
drop_cols = ['City', 'Datetime', 'AQI_Bucket', 'Hour']

# Separate features (X) and target variable (y)
X = final_df.drop(columns=['AQI'] + drop_cols, errors='ignore')
y = final_df['AQI']

# Split into Train and Test sets
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"📊 Train features shape: {X_train.shape}")
print(f"📉 Test features shape: {X_test.shape}")
print("\nFeatures your model will actively learn from:")
print(X_train.columns.tolist())

🔄 Step 5: Applying Cyclical Encoding to Datetime...
📅 Step 6: Splitting data chronologically...
📊 Train features shape: (14516, 19)
📉 Test features shape: (3629, 19)

Features your model will actively learn from:
['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'temperature', 'humidity', 'wind_speed', 'traffic_density', 'industrial_activity', 'AQI_lag_1h', 'AQI_lag_2h', 'AQI_lag_24h', 'Hour_sin', 'Hour_cos']
